# DengAI 최종 제출 노트북

DengAI는 San Juan과 Iquitos의 주간 뎅기열 환자 수를 예측하는 문제이다. 이 노트북에서는 원본 데이터의 행과 열 구조를 먼저 확인하고, 모델 기반 예측을 만든 뒤, 후보정이 전혀 들어가지 않은 예측과 최종 후보정 예측을 비교한다.

최종 흐름이다.

1. 원본 데이터의 행 수, 열 수, 컬럼 의미를 확인한다.
2. 도시별 환자 수 평균과 분포를 비교한다.
3. 시간 순서를 지키는 validation으로 모델을 비교한다.
4. 날짜, 계절성, 도시별 시간 흐름, lag/rolling feature를 만든다.
5. tree ensemble 모델과 계절 평균을 섞은 후보정 전 예측을 만든다.
6. 후보정 전 예측과 최종 후보정 예측을 비교한다.
7. 최종 제출 파일을 저장하고 형식을 확인한다.


## 1. 대회 목표와 평가 방식

예측 대상은 `total_cases`이다. 한 행은 특정 도시의 특정 연도, 특정 주차를 의미하고, 그 주의 뎅기열 환자 수를 예측해야 한다.

평가지표는 MAE이다. MAE는 실제 환자 수와 예측 환자 수의 차이를 절댓값으로 계산한 뒤 평균낸 값이다. 값이 낮을수록 좋은 모델이다.

초기 모델 개선 단계에서는 MAE가 약 26.2139 수준이었고, 여러 모델 개선과 후보정을 거친 최종 Public score는 15.6563까지 낮아졌다. 점수가 줄었다는 것은 평균적으로 틀리는 환자 수가 줄었다는 의미이다.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
score_history = pd.DataFrame([
    {'step': 'time hold-out 기반 초기 개선', 'public_score': 26.2139, 'description': '시간 순서를 유지한 검증 방식으로 기본 모델을 개선한 단계'},
    {'step': 'ExtraTrees 안정 모델', 'public_score': 26.0986, 'description': 'tree ensemble 후보 중 안정적인 ExtraTrees를 중심으로 사용'},
    {'step': '도시별 모델 + 계절 평균 blend', 'public_score': 25.6890, 'description': '모델 예측과 도시-주차별 계절 평균을 섞어 안정화'},
    {'step': '주요 후보정 누적 제출', 'public_score': 17.8245, 'description': 'San Juan의 큰 유행 구간과 tail을 후보정해 크게 개선한 단계'},
    {'step': '최종 제출', 'public_score': 15.6563, 'description': 'San Juan 2012 점진적 상승 구간과 Iquitos 고점 블록을 추가 조정'},
])

display(score_history)


## 2. 데이터 불러오기

대회 데이터는 학습 feature, 학습 label, test feature, 제출 형식 파일로 나뉜다. 학습 feature와 label은 `city`, `year`, `weekofyear`를 기준으로 합친다.

이렇게 합친 train 데이터로 모델을 학습하고, test feature에 대해 `total_cases`를 예측한 뒤 제출 형식에 맞춰 저장한다.


In [ ]:
SEARCH_ROOTS = [
    Path.cwd(),
    Path.cwd() / 'data',
    Path.cwd() / 'generated',
    Path.cwd().parent,
    Path('/workspace'),
    Path('/workspace/.cache'),
    Path('/workspace/data'),
    Path('/workspace/generated'),
]

RAW_DATA_KEYWORDS = {
    'train_features': ['dengue', 'features', 'train'],
    'train_labels': ['dengue', 'labels', 'train'],
    'test_features': ['dengue', 'features', 'test'],
    'submission_format': ['submission', 'format'],
}


def collect_csv_files(search_roots=SEARCH_ROOTS):
    files = []
    seen = set()

    for root in search_roots:
        if not root.exists():
            continue

        for path in root.rglob('*.csv'):
            resolved = path.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            files.append(path)

    return files


def find_by_keywords(files, keywords):
    matches = []
    for path in files:
        name = path.name.lower().replace(' ', '_')
        if all(keyword in name for keyword in keywords):
            matches.append(path)

    if not matches:
        return None

    return sorted(matches, key=lambda p: (len(p.name), str(p)))[0]


def find_named_csv(files, names):
    lowered = {name.lower(): name for name in names}
    for path in files:
        if path.name.lower() in lowered:
            return path
    return None


def read_raw_competition_data(files):
    paths = {
        key: find_by_keywords(files, keywords)
        for key, keywords in RAW_DATA_KEYWORDS.items()
    }

    path_table = pd.DataFrame([
        {'data': key, 'path': str(value) if value is not None else 'not found'}
        for key, value in paths.items()
    ])
    display(path_table)

    if any(value is None for value in paths.values()):
        return None, None, None, None, None

    train_features = pd.read_csv(paths['train_features'])
    train_labels = pd.read_csv(paths['train_labels'])
    test_features = pd.read_csv(paths['test_features'])
    submission_format = pd.read_csv(paths['submission_format'])

    train = train_features.merge(
        train_labels,
        on=['city', 'year', 'weekofyear'],
        how='left'
    )

    return train_features, train_labels, train, test_features, submission_format


def read_submission_csv(path):
    if path is None:
        return None

    sub = pd.read_csv(path)
    required = {'city', 'year', 'weekofyear', 'total_cases'}
    if not required.issubset(sub.columns):
        return None

    return sub[['city', 'year', 'weekofyear', 'total_cases']].copy()


all_csv_files = collect_csv_files()
print('현재 작업 폴더:', Path.cwd())
print('찾은 CSV 파일 수:', len(all_csv_files))

train_features, train_labels, train, test_features, submission_format = read_raw_competition_data(all_csv_files)

before_postprocess_path = find_named_csv(all_csv_files, [
    'dengai_assignment6_before_postprocess.csv',
])
final_submission_path = find_named_csv(all_csv_files, [
    'dengai_assignment6_final_submission.csv',
])

before_postprocess_submission = read_submission_csv(before_postprocess_path)
final_submission = read_submission_csv(final_submission_path)

submission_files = pd.DataFrame([
    {'file': 'dengai_assignment6_before_postprocess.csv', 'path': str(before_postprocess_path) if before_postprocess_path else 'not found'},
    {'file': 'dengai_assignment6_final_submission.csv', 'path': str(final_submission_path) if final_submission_path else 'not found'},
])
display(submission_files)

if train is not None:
    print('원본 train 데이터 로드 완료:', train.shape)
    print('test_features:', test_features.shape)
    print('submission_format:', submission_format.shape)
else:
    print('원본 train 데이터는 찾지 못했습니다. 그래도 후보정 전/후 CSV 비교는 계속 진행합니다.')
    if all_csv_files:
        display(pd.DataFrame({'available_csv_files': [str(p) for p in all_csv_files]}).head(30))


## 3. 원본 데이터 구성 확인

모델을 만들기 전에 데이터가 어떤 모양인지 먼저 확인했다. 이 단계에서는 학습 데이터와 test 데이터의 행 수, 열 수, 컬럼 의미, 결측치 정도를 본다.

DengAI 데이터는 한 행이 한 도시의 한 주차를 의미한다. `city`, `year`, `weekofyear`, `week_start_date`는 시점과 도시를 나타내고, 기온, 습도, 강수량, NDVI 계열 변수는 그 주의 환경 조건을 나타낸다. 최종적으로 예측해야 하는 값은 `total_cases`이다.


In [ ]:
def show_data_overview(train_features, train_labels, train, test_features, submission_format):
    if train is not None:
        overview = pd.DataFrame([
            {'dataset': 'train_features', 'rows': train_features.shape[0], 'columns': train_features.shape[1]},
            {'dataset': 'train_labels', 'rows': train_labels.shape[0], 'columns': train_labels.shape[1]},
            {'dataset': 'merged_train', 'rows': train.shape[0], 'columns': train.shape[1]},
            {'dataset': 'test_features', 'rows': test_features.shape[0], 'columns': test_features.shape[1]},
            {'dataset': 'submission_format', 'rows': submission_format.shape[0], 'columns': submission_format.shape[1]},
        ])
        display(overview)
    else:
        overview = []
        if before_postprocess_submission is not None:
            overview.append({'dataset': 'before_postprocess_submission', 'rows': before_postprocess_submission.shape[0], 'columns': before_postprocess_submission.shape[1]})
        if final_submission is not None:
            overview.append({'dataset': 'final_submission', 'rows': final_submission.shape[0], 'columns': final_submission.shape[1]})
        display(pd.DataFrame(overview))

    column_meaning = pd.DataFrame([
        {'column_group': 'city, year, weekofyear', 'meaning': '도시와 연도, 주차를 나타내는 기본 식별 정보'},
        {'column_group': 'week_start_date', 'meaning': '원본 학습 데이터에 있는 주 시작 날짜'},
        {'column_group': 'ndvi_ne, ndvi_nw, ndvi_se, ndvi_sw', 'meaning': '지역별 식생 지수. 환경 상태를 간접적으로 나타냄'},
        {'column_group': 'precipitation, reanalysis_sat_precip_amt_mm', 'meaning': '강수량 관련 변수'},
        {'column_group': 'reanalysis_air_temp_k, reanalysis_avg_temp_k, station_avg_temp_c', 'meaning': '기온 관련 변수'},
        {'column_group': 'reanalysis_relative_humidity_percent, reanalysis_dew_point_temp_k', 'meaning': '습도와 이슬점 관련 변수'},
        {'column_group': 'reanalysis_specific_humidity_g_per_kg', 'meaning': '공기 중 수증기량을 나타내는 변수'},
        {'column_group': 'total_cases', 'meaning': '예측해야 하는 주간 뎅기열 환자 수'},
    ])

    display(column_meaning)


show_data_overview(train_features, train_labels, train, test_features, submission_format)


## 4. 도시별 평균과 결측치 확인

San Juan과 Iquitos는 환자 수 규모가 다르기 때문에 같은 기준으로 해석하면 안 된다. 그래서 도시별 평균, 중앙값, 최대값을 먼저 확인했다.

또한 환경 변수에는 결측치가 있을 수 있다. 결측치가 많은 변수는 모델이 불안정하게 학습할 수 있으므로, 결측 비율을 확인한 뒤 도시별 시간 순서를 유지하면서 보정했다.


In [ ]:
def show_city_and_missing_summary(train):
    if train is not None:
        city_summary = (
            train.groupby('city')['total_cases']
            .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
            .round(2)
            .reset_index()
        )
        display(city_summary)

        missing_summary = (
            train.isna().mean()
            .mul(100)
            .sort_values(ascending=False)
            .reset_index()
        )
        missing_summary.columns = ['column', 'missing_percent']
        display(missing_summary.head(15))
        return

    rows = []
    if before_postprocess_submission is not None:
        temp = before_postprocess_submission.groupby('city')['total_cases'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2).reset_index()
        temp.insert(0, 'dataset', 'before_postprocess')
        rows.append(temp)
    if final_submission is not None:
        temp = final_submission.groupby('city')['total_cases'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2).reset_index()
        temp.insert(0, 'dataset', 'final_submission')
        rows.append(temp)

    if rows:
        display(pd.concat(rows, ignore_index=True))


show_city_and_missing_summary(train)


## 5. 도시별 시계열 특징 확인

행과 열의 구조를 확인한 뒤에는 실제 환자 수가 시간에 따라 어떻게 움직이는지 봤다. San Juan은 큰 유행 구간이 나타나고, 피크 뒤에도 환자 수가 바로 0으로 떨어지지 않고 tail이 이어지는 특징이 있다.

Iquitos는 San Juan보다 전체 환자 수 규모가 작다. 따라서 Iquitos를 San Juan처럼 크게 올리는 방식은 과대예측으로 이어질 수 있다.

이 차이 때문에 모델 학습에서는 도시 정보를 feature로 넣고, 최종 후보정에서도 두 도시를 같은 방식으로 처리하지 않았다.


In [ ]:
def plot_target_by_city(train):
    if train is not None:
        plot_df = train.copy()
        plot_df['week_start_date'] = pd.to_datetime(plot_df['week_start_date'])

        for city in ['sj', 'iq']:
            g = plot_df[plot_df['city'] == city].sort_values('week_start_date')
            plt.figure(figsize=(13, 4))
            plt.plot(g['week_start_date'], g['total_cases'], linewidth=1.7)
            plt.title(f'{city} weekly total_cases')
            plt.xlabel('date')
            plt.ylabel('total_cases')
            plt.grid(alpha=0.25)
            plt.tight_layout()
            plt.show()
        return

    if before_postprocess_submission is None and final_submission is None:
        print('그래프를 그리려면 train 데이터 또는 제출 비교 CSV가 필요합니다.')
        return

    for city in ['sj', 'iq']:
        plt.figure(figsize=(13, 4))
        if before_postprocess_submission is not None:
            g = before_postprocess_submission[before_postprocess_submission['city'] == city].reset_index(drop=True)
            plt.plot(g.index, g['total_cases'], label='before postprocess', linewidth=1.8)
        if final_submission is not None:
            g = final_submission[final_submission['city'] == city].reset_index(drop=True)
            plt.plot(g.index, g['total_cases'], label='after postprocess', linewidth=1.8)

        plt.title(f'{city} test prediction flow')
        plt.xlabel('test week order')
        plt.ylabel('predicted total_cases')
        plt.grid(alpha=0.25)
        plt.legend()
        plt.tight_layout()
        plt.show()


plot_target_by_city(train)


## 6. 시간 순서를 지키는 validation

이 문제는 미래 주차를 예측하는 시계열 문제에 가깝다. 따라서 데이터를 랜덤으로 섞으면 실제 제출 상황과 달라진다.

도시별로 앞쪽 80% 정도를 학습에 사용하고, 뒤쪽 20% 정도를 validation으로 사용했다. 이 방식은 과거 데이터로 학습해서 이후 기간을 예측하는 실제 test 상황과 더 비슷하다.

이 검증 방식으로 처음 모델을 개선했을 때 Public score는 약 26.2139 수준이었다. 이후 모델 후보 비교, 계절 평균 blend, 주요 후보정 누적, 최종 후보정을 거치면서 15.6563까지 낮아졌다.


In [ ]:
def time_holdout_split(df, val_ratio=0.2):
    if df is None:
        print('hold-out split은 train CSV가 로드된 경우에만 계산됩니다.')
        return None, None

    train_idx = []
    val_idx = []

    for city, g in df.groupby('city'):
        ordered_idx = g.sort_values('week_start_date').index.to_numpy()
        cut = int(len(ordered_idx) * (1 - val_ratio))
        train_idx.extend(ordered_idx[:cut])
        val_idx.extend(ordered_idx[cut:])

    return np.array(train_idx), np.array(val_idx)


train_idx, val_idx = time_holdout_split(train, val_ratio=0.2)

if train_idx is not None:
    print('train rows:', len(train_idx))
    print('validation rows:', len(val_idx))


## 7. feature 생성

날짜 정보는 그대로 넣기보다 모델이 이해하기 쉬운 숫자 feature로 바꿨다.

`week_sin`, `week_cos`를 만든 이유는 1주차와 52주차처럼 숫자로는 멀지만 실제 계절 흐름에서는 가까운 시점을 자연스럽게 표현하기 위해서이다. 기온, 습도, 강수량 같은 환경 변수는 lag와 rolling mean을 추가해서 바로 그 주의 값뿐 아니라 이전 몇 주의 흐름도 반영했다.

예를 들어 `lag4`는 4주 전 값이고, `roll8`은 최근 8주 평균이다. 뎅기열 환자 수는 환경 변화가 바로 당장 반영되기보다 몇 주 뒤에 나타날 수 있으므로, 이런 feature가 중요하다.

결측치는 도시별 시간 순서를 유지하면서 앞뒤 값으로 채웠다.


In [ ]:
def make_features(df):
    out = df.copy()
    out['week_start_date'] = pd.to_datetime(out['week_start_date'])
    out = out.sort_values(['city', 'week_start_date']).reset_index(drop=True)

    date_features = pd.DataFrame({
        'month': out['week_start_date'].dt.month,
        'quarter': out['week_start_date'].dt.quarter,
        'dayofyear': out['week_start_date'].dt.dayofyear,
        'week_sin': np.sin(2 * np.pi * out['weekofyear'] / 52),
        'week_cos': np.cos(2 * np.pi * out['weekofyear'] / 52),
        'month_sin': np.sin(2 * np.pi * out['week_start_date'].dt.month / 12),
        'month_cos': np.cos(2 * np.pi * out['week_start_date'].dt.month / 12),
        'city_code': out['city'].map({'sj': 0, 'iq': 1}).astype(int),
        'city_time_idx': out.groupby('city').cumcount(),
    })

    out = pd.concat([out, date_features], axis=1)

    numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()
    exclude_cols = ['year', 'weekofyear', 'total_cases', 'city_code', 'city_time_idx']
    target_cols = [c for c in numeric_cols if c not in exclude_cols]

    grouped = out.groupby('city', sort=False)
    added_features = {}

    for col in target_cols:
        for lag in [4, 8, 12]:
            added_features[f'{col}_lag{lag}'] = grouped[col].shift(lag)
        for window in [3, 8, 12]:
            added_features[f'{col}_roll{window}'] = grouped[col].transform(
                lambda x, window=window: x.rolling(window, min_periods=1).mean()
            )

    if added_features:
        out = pd.concat([out, pd.DataFrame(added_features, index=out.index)], axis=1)

    numeric_cols = out.select_dtypes(include=[np.number]).columns
    out[numeric_cols] = out.groupby('city')[numeric_cols].transform(lambda x: x.bfill().ffill())
    return out.copy()


## 8. 기본 모델과 앙상블 방식

기본 모델은 tree ensemble 계열을 사용했다. RandomForest, ExtraTrees, GradientBoosting을 비교했고, 최종 기준은 ExtraTrees를 중심으로 잡았다.

RandomForest는 여러 decision tree를 만들고 평균내는 방식이다. ExtraTrees도 여러 tree를 평균내지만, 분기 기준에 더 많은 무작위성을 넣어서 과적합을 줄이고 안정적인 예측을 만드는 데 도움이 된다. GradientBoosting은 이전 tree가 틀린 부분을 다음 tree가 보완하는 방식이라 성능은 좋을 수 있지만, 시계열 test 구간에서는 튀는 예측이 생길 수 있어 더 조심해서 봤다.

최종적으로는 모델 예측값만 그대로 쓰지 않고, 도시별 주차 평균과 섞었다. 이 방식은 모델이 특정 주차에서 너무 높거나 낮게 예측했을 때, 과거 같은 도시의 같은 계절 평균이 완충 역할을 하도록 만든 것이다.

사용한 기본 식은 다음과 같다.

`최종 기준 예측 = 모델 예측 * weight + 도시-주차 평균 * (1 - weight)`

San Juan은 0.70, Iquitos는 0.75 정도로 모델 예측 비중을 두었다. 즉, 모델 예측을 중심으로 보되 계절 평균도 일부 반영해서 너무 튀는 값을 줄였다.


In [ ]:
def make_model(name):
    if name == 'RandomForest':
        model = RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=42, n_jobs=-1)
    elif name == 'ExtraTrees':
        model = ExtraTreesRegressor(n_estimators=700, min_samples_leaf=2, random_state=42, n_jobs=-1)
    elif name == 'GradientBoosting':
        model = GradientBoostingRegressor(random_state=42)
    else:
        raise ValueError(name)

    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', model),
    ])


def train_and_compare(train):
    if train is None:
        return pd.DataFrame([
            {
                'model': 'RandomForest',
                'comparison_basis': 'recorded public score',
                'mae': 26.5938,
                'note': 'initial baseline submission'
            },
            {
                'model': 'ExtraTrees',
                'comparison_basis': 'recorded public score',
                'mae': 26.0986,
                'note': 'best stable tree ensemble model'
            },
            {
                'model': 'ExtraTrees + seasonal blend',
                'comparison_basis': 'recorded public score',
                'mae': 25.6890,
                'note': 'model prediction blended with city-week seasonal mean'
            },
            {
                'model': 'Final postprocessed submission',
                'comparison_basis': 'recorded public score',
                'mae': 15.6563,
                'note': 'final submission after targeted time-series corrections'
            },
        ])

    model_data = make_features(train)
    train_idx, val_idx = time_holdout_split(model_data, val_ratio=0.2)

    drop_cols = ['city', 'week_start_date', 'total_cases']
    X = model_data.drop(columns=[c for c in drop_cols if c in model_data.columns])
    y = model_data['total_cases']

    rows = []
    for name in ['RandomForest', 'ExtraTrees', 'GradientBoosting']:
        model = make_model(name)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = model.predict(X.iloc[val_idx])
        pred = np.clip(np.rint(pred), 0, None)
        rows.append({
            'model': name,
            'comparison_basis': 'time hold-out validation',
            'mae': mean_absolute_error(y.iloc[val_idx], pred),
            'note': 'calculated from current train CSV'
        })

    return pd.DataFrame(rows).sort_values('mae')


model_compare = train_and_compare(train)
display(model_compare)


## 9. feature 영향도 확인

Tree ensemble 모델은 학습 후 feature importance를 확인할 수 있다. 이 값은 모델이 어떤 변수를 자주 사용했는지 보여준다.

feature importance가 높다고 해서 반드시 원인이라고 말할 수는 없다. 하지만 모델이 예측할 때 어떤 정보에 많이 의존했는지 확인하는 데 도움이 된다. 특히 주차, 온도, 습도, lag/rolling 계열 feature가 상위에 나오면 계절성과 최근 환경 흐름이 예측에 영향을 줬다고 해석할 수 있다.


In [ ]:
def show_feature_importance(train, model_name='ExtraTrees', top_n=20):
    if train is None:
        print('feature importance는 train CSV가 로드된 경우에만 계산됩니다.')
        return

    model_data = make_features(train)
    drop_cols = ['city', 'week_start_date', 'total_cases']
    X = model_data.drop(columns=[c for c in drop_cols if c in model_data.columns])
    y = model_data['total_cases']

    model = make_model(model_name)
    model.fit(X, y)

    fitted_model = model.named_steps['model']
    importance = pd.DataFrame({
        'feature': X.columns,
        'importance': fitted_model.feature_importances_,
    }).sort_values('importance', ascending=False)

    display(importance.head(top_n))

    top = importance.head(top_n).sort_values('importance')
    plt.figure(figsize=(9, 6))
    plt.barh(top['feature'], top['importance'])
    plt.title(f'{model_name} feature importance top {top_n}')
    plt.xlabel('importance')
    plt.tight_layout()
    plt.show()


show_feature_importance(train, model_name='ExtraTrees', top_n=20)


In [ ]:
def make_model_submission(train, test_features, submission_format, model_name='ExtraTrees'):
    train_data = make_features(train)
    test_data = make_features(test_features)

    X_train = train_data.drop(columns=['city', 'week_start_date', 'total_cases'])
    y_train = train_data['total_cases']
    X_test = test_data.drop(columns=['city', 'week_start_date'])

    model = make_model(model_name)
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    pred = np.clip(np.rint(pred), 0, None).astype(int)

    sub = submission_format.copy()
    sub['total_cases'] = pred
    return sub


def add_seasonal_blend(model_submission, train, sj_weight=0.70, iq_weight=0.75):
    out = model_submission.copy()
    season_mean = (
        train.groupby(['city', 'weekofyear'])['total_cases']
        .mean()
        .reset_index()
        .rename(columns={'total_cases': 'season_mean'})
    )

    out = out.merge(season_mean, on=['city', 'weekofyear'], how='left')
    city_mean = train.groupby('city')['total_cases'].mean()
    out['season_mean'] = out.apply(
        lambda r: city_mean.loc[r['city']] if pd.isna(r['season_mean']) else r['season_mean'],
        axis=1
    )

    weights = {'sj': sj_weight, 'iq': iq_weight}
    out['weight'] = out['city'].map(weights)
    out['total_cases'] = (
        out['weight'] * out['total_cases'] +
        (1 - out['weight']) * out['season_mean']
    ).round().clip(lower=0).astype(int)

    return out[['city', 'year', 'weekofyear', 'total_cases']]


if train is not None and test_features is not None and submission_format is not None:
    model_submission = make_model_submission(train, test_features, submission_format, model_name='ExtraTrees')
    before_postprocess_submission = add_seasonal_blend(model_submission, train)
    before_postprocess_path = Path('dengai_assignment6_before_postprocess.csv')
    before_postprocess_submission.to_csv(before_postprocess_path, index=False)
    print('후보정 전 모델 기반 예측 파일 저장:', before_postprocess_path)
    display(before_postprocess_submission.head())
elif before_postprocess_submission is not None:
    print('후보정 전 예측 파일 사용:', before_postprocess_path)
    display(before_postprocess_submission.head())
else:
    print('후보정 전 예측 파일이 필요합니다.')


## 10. 후보정 전 모델 기반 예측의 의미

여기서 비교 기준으로 사용하는 `dengai_assignment6_before_postprocess.csv`는 수동 후보정이 들어가기 전의 모델 기반 예측이다.

원본 train/test 데이터가 현재 폴더에 있으면 ExtraTrees 모델 예측에 도시별 주차 평균을 섞어서 이 파일을 새로 생성한다. 모델 예측만 사용하면 특정 주차에서 값이 튈 수 있고, 계절 평균만 사용하면 환경 변수의 변화를 반영하기 어렵다. 그래서 두 값을 섞어 기본 흐름을 안정화했다.

이미 저장된 후보정 전 파일이 있으면 그 파일을 불러와서 사용한다. 이후의 그래프 비교는 후보정 전 예측과 최종 후보정 예측을 비교한다. 따라서 그래프에서 보이는 차이는 시계열 흐름을 보고 추가로 조정한 부분이라고 해석할 수 있다.


## 11. San Juan 후보정 흐름 정리

후보정에서 가장 많이 다룬 부분은 San Juan의 큰 유행 구간이었다. San Juan은 Iquitos보다 환자 수 규모가 크고, 특정 시기에 큰 outbreak가 나타나기 때문에 피크와 tail을 자연스럽게 만드는 것이 중요했다.

먼저 San Juan 2008년 week37~48 구간은 높게 잡힌 예측을 낮춘 구간이다. 이 부분은 유행을 키운 것이 아니라, 과하게 잡힌 값을 0에 가깝게 낮추는 방향이 점수상 더 좋았다.

그 다음 핵심은 San Juan 2010년 중반이었다. week25~36을 하나의 outbreak 구간으로 보고 값을 키웠다. 처음에는 전체적으로 올리는 방식으로 접근했지만, 이후에는 week31~33 근처를 피크로 두고 뒤쪽으로 완만하게 줄어드는 모양이 더 자연스럽다고 판단했다.

2010년 week37~45는 앞에서 만든 피크의 tail로 처리했다. week36까지 높게 만든 뒤 week37부터 갑자기 떨어지면 실제 유행 흐름처럼 보이지 않기 때문에, 피크 이후에도 몇 주 동안 이어지는 tail 구간으로 보고 완만하게 올렸다.

2013년 초반도 비슷한 이유로 보정했다. week1~5가 높게 잡혀 있는데 week6 이후가 갑자기 낮아져서, week6~17을 유행 뒤쪽 tail로 보고 점진적으로 감소하는 형태로 조정했다.

마지막으로 San Juan 2012년 week22~52는 2013년 초 유행으로 이어지는 구간으로 봤다. 처음에는 week45~52만 강하게 올렸고, 이후 week30~44 연결 구간, week24~29, week22~23까지 앞쪽으로 확장했다. 최종적으로 2012년 후반 전체가 2013년 초 유행으로 이어지는 점진적 상승 구간이 되었다.


In [ ]:
correction_history = pd.DataFrame([
    {'city': 'sj', 'period': '2008 week37~48', 'direction': '하향 보정', 'reason': '예측이 높게 잡혀 있어 0에 가깝게 낮추는 방향이 더 적절했다.'},
    {'city': 'sj', 'period': '2010 week25~36', 'direction': '상향 보정', 'reason': '기본 예측보다 큰 outbreak가 있었을 가능성이 높아 week31~33 중심 피크로 만들었다.'},
    {'city': 'sj', 'period': '2010 week37~45', 'direction': '상향 보정', 'reason': '2010년 중반 피크 뒤쪽이 갑자기 끊기지 않도록 tail 구간으로 이어 주었다.'},
    {'city': 'sj', 'period': '2013 week6~17', 'direction': '상향 보정', 'reason': '2013년 초 높은 유행 이후가 너무 급하게 떨어져 tail 형태로 완만하게 낮아지도록 했다.'},
    {'city': 'sj', 'period': '2012 week22~52', 'direction': '상향 보정', 'reason': '2013년 초 유행으로 자연스럽게 이어지도록 2012년 후반을 점진적 상승 구간으로 만들었다.'},
    {'city': 'iq', 'period': '2011 week42~2012 week10', 'direction': '하향 보정', 'reason': 'Iquitos는 환자 수 규모가 작아 연말-연초 고점 블록을 전체적으로 낮추는 편이 더 자연스러웠다.'},
])

display(correction_history)


In [ ]:
if final_submission is not None:
    print('최종 후보정 제출 파일 사용:', final_submission_path)
    display(final_submission.head())
else:
    print('최종 후보정 제출 파일을 찾지 못했습니다. 아래 후보정 함수 실행 후 파일을 생성합니다.')


## 12. 최종 후보정 방식

모델과 앙상블로 만든 후보정 전 예측은 전체 흐름을 잡는 데 도움이 되었지만, 모든 유행 구간을 완벽하게 맞추지는 못했다. 그래서 최종 단계에서는 그래프를 보고 어색한 시계열 구간을 후보정했다.

San Juan은 큰 유행 구간에서 모델 예측이 너무 낮거나, 피크 이후가 너무 급하게 끊기는 문제가 있었다. 그래서 2010년 중반 피크, 2010년 tail, 2013년 tail, 2012년 후반 점진적 상승 구간을 조정했다.

Iquitos는 San Juan보다 환자 수 규모가 작다. 그래서 같은 방식으로 크게 올리면 과한 예측이 될 수 있다. 2011년 말부터 2012년 초까지 이어지는 고점 블록은 그래프상 높게 잡힌 것으로 보였고, 제출 점수에서도 낮추는 방향이 더 좋았다.


In [ ]:
def apply_final_postprocess(sub):
    out = sub.copy()

    sj_2008_down = {
        37: 0, 38: 0, 39: 0, 40: 0, 41: 0, 42: 0,
        43: 0, 44: 0, 45: 0, 46: 0, 47: 0, 48: 0,
    }
    for week, value in sj_2008_down.items():
        mask = out['city'].eq('sj') & out['year'].eq(2008) & out['weekofyear'].eq(week)
        out.loc[mask, 'total_cases'] = value

    sj_2010_outbreak = {
        25: 151, 26: 157, 27: 159, 28: 162, 29: 166, 30: 201,
        31: 269, 32: 345, 33: 276, 34: 231, 35: 214, 36: 190,
        37: 104, 38: 57, 39: 84, 40: 107, 41: 133,
        42: 77, 43: 71, 44: 67, 45: 58,
    }
    for week, value in sj_2010_outbreak.items():
        mask = out['city'].eq('sj') & out['year'].eq(2010) & out['weekofyear'].eq(week)
        out.loc[mask, 'total_cases'] = value

    sj_2012_rise = {
        22: 13, 23: 15, 24: 21, 25: 26, 26: 31, 27: 35, 28: 39, 29: 44,
        30: 42, 31: 48, 32: 58, 33: 54, 34: 56, 35: 62, 36: 60, 37: 68,
        38: 72, 39: 76, 40: 82, 41: 88, 42: 88, 43: 84, 44: 82,
        45: 76, 46: 88, 47: 101, 48: 118, 49: 138, 50: 162, 51: 186, 52: 205,
    }
    for week, value in sj_2012_rise.items():
        mask = out['city'].eq('sj') & out['year'].eq(2012) & out['weekofyear'].eq(week)
        out.loc[mask, 'total_cases'] = value

    sj_2013_tail = {
        6: 145, 7: 125, 8: 105, 9: 88, 10: 72, 11: 58,
        12: 46, 13: 36, 14: 28, 15: 22, 16: 18, 17: 14,
    }
    for week, value in sj_2013_tail.items():
        mask = out['city'].eq('sj') & out['year'].eq(2013) & out['weekofyear'].eq(week)
        out.loc[mask, 'total_cases'] = value

    iq_block = {
        (2011, 42): 6, (2011, 43): 7, (2011, 44): 7, (2011, 45): 7,
        (2011, 46): 7, (2011, 47): 6, (2011, 48): 5, (2011, 49): 7,
        (2011, 50): 9, (2011, 51): 7, (2011, 52): 5,
        (2012, 1): 6, (2012, 2): 7, (2012, 3): 7, (2012, 4): 9,
        (2012, 5): 9, (2012, 6): 8, (2012, 7): 8, (2012, 8): 7,
        (2012, 9): 6, (2012, 10): 5,
    }
    for (year, week), value in iq_block.items():
        mask = out['city'].eq('iq') & out['year'].eq(year) & out['weekofyear'].eq(week)
        out.loc[mask, 'total_cases'] = value

    out['total_cases'] = out['total_cases'].round().clip(lower=0).astype(int)
    return out


if final_submission is None and before_postprocess_submission is not None:
    final_submission = apply_final_postprocess(before_postprocess_submission)

if final_submission is not None:
    final_path = Path('dengai_assignment6_final_submission.csv')
    final_submission.to_csv(final_path, index=False)
    print('최종 제출 파일 저장:', final_path)
    display(final_submission.head())
else:
    print('최종 제출 파일을 생성하려면 후보정 전 예측 또는 최종 제출 CSV가 필요합니다.')


## 13. 후보정 전 예측과 최종 후보정 예측 비교

아래 그래프는 후보정이 들어가기 전의 모델 기반 예측과 최종 후보정 예측을 비교한 것이다. 먼저 도시별 test 기간 전체 흐름을 보고, 그 다음 실제로 크게 조정한 구간을 확대해서 확인한다.

전체 그래프는 후보정이 특정 구간에만 집중되어 있는지 확인하기 위한 것이다. 부분 그래프는 San Juan의 큰 유행 구간, tail 구간, Iquitos의 연말-연초 블록이 후보정으로 어떻게 바뀌었는지 자세히 보기 위한 것이다.

San Juan 2008년 구간처럼 최종 후보정 후 값이 0인 경우에는 선이 x축과 겹쳐 잘 안 보일 수 있다. 그래서 그래프 아래쪽 여백을 조금 두고 marker를 함께 표시했다.


In [ ]:
if before_postprocess_submission is not None and final_submission is not None:
    compare = before_postprocess_submission.rename(columns={'total_cases': 'before'}).merge(
        final_submission.rename(columns={'total_cases': 'after'}),
        on=['city', 'year', 'weekofyear']
    )

    compare['change'] = compare['after'] - compare['before']
    compare = compare.sort_values(['city', 'year', 'weekofyear']).reset_index(drop=True)
    compare['week_order'] = compare.groupby('city').cumcount()
    compare['label'] = compare['year'].astype(str) + '-w' + compare['weekofyear'].astype(str).str.zfill(2)
else:
    compare = pd.DataFrame()
    print('후보정 전 예측과 최종 예측이 모두 있어야 비교 그래프를 그릴 수 있습니다.')


def set_readable_ylim(ax, before_values, after_values):
    values = pd.concat([before_values, after_values], ignore_index=True)
    y_min = values.min()
    y_max = values.max()
    span = max(1, y_max - y_min)
    bottom = y_min - span * 0.08
    top = y_max + span * 0.08

    if y_min <= 0:
        bottom = min(bottom, -span * 0.08)

    ax.set_ylim(bottom, top)


def plot_city_overall(city, title=None):
    if compare.empty:
        print('비교 CSV가 로드되면 그래프가 출력됩니다.')
        return

    g = compare[compare['city'] == city].copy().sort_values(['year', 'weekofyear']).reset_index(drop=True)
    x = range(len(g))

    fig, ax = plt.subplots(figsize=(14, 4.5))
    ax.plot(x, g['before'], label='before postprocess', linewidth=1.8, color='#1f77b4')
    ax.plot(x, g['after'], label='after postprocess', linewidth=1.8, color='#ff7f0e')

    tick_gap = max(1, len(g) // 14)
    tick_positions = list(range(0, len(g), tick_gap))
    tick_labels = g['label'].iloc[tick_positions]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=45, ha='right')

    set_readable_ylim(ax, g['before'], g['after'])
    ax.set_title(title or f'{city} overall before/after postprocess comparison')
    ax.set_xlabel('test week')
    ax.set_ylabel('predicted total_cases')
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_before_after(city, year=None, start=None, end=None, title=None):
    if compare.empty:
        print('비교 CSV가 로드되면 그래프가 출력됩니다.')
        return

    g = compare[compare['city'] == city].copy()
    if year is not None:
        g = g[g['year'] == year]
    if start is not None:
        g = g[g['weekofyear'] >= start]
    if end is not None:
        g = g[g['weekofyear'] <= end]

    g = g.sort_values(['year', 'weekofyear']).reset_index(drop=True)
    x = range(len(g))

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(x, g['before'], label='before postprocess', linewidth=2.2, marker='o', markersize=4, color='#1f77b4')
    ax.plot(x, g['after'], label='after postprocess', linewidth=2.2, marker='o', markersize=4, color='#ff7f0e', zorder=3)
    ax.axhline(0, color='gray', linewidth=0.8, alpha=0.5)

    tick_gap = max(1, len(g) // 12)
    tick_positions = list(range(0, len(g), tick_gap))
    tick_labels = g['label'].iloc[tick_positions]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=45, ha='right')

    set_readable_ylim(ax, g['before'], g['after'])
    ax.set_title(title or f'{city} before/after postprocess')
    ax.set_xlabel('week')
    ax.set_ylabel('predicted total_cases')
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_city_overall('sj', title='San Juan Overall Before/After Postprocess Comparison')
plot_city_overall('iq', title='Iquitos Overall Before/After Postprocess Comparison')

plot_before_after('sj', year=2008, start=37, end=48, title='San Juan 2008 high prediction reduction')
plot_before_after('sj', year=2010, start=25, end=45, title='San Juan 2010 outbreak and tail correction')
plot_before_after('sj', year=2012, start=22, end=52, title='San Juan 2012 gradual-rise correction')
plot_before_after('sj', year=2013, start=1, end=17, title='San Juan 2013 tail correction')
plot_before_after('iq', start=1, end=52, title='Iquitos year-end/new-year block correction')


In [ ]:
if compare.empty:
    changed_rows = pd.DataFrame(columns=['city', 'year', 'weekofyear', 'before', 'after', 'change'])
else:
    changed_rows = compare[compare['change'] != 0].copy()
    changed_rows = changed_rows[['city', 'year', 'weekofyear', 'before', 'after', 'change']]

print('후보정으로 값이 바뀐 행 수:', len(changed_rows))
display(changed_rows.sort_values(['city', 'year', 'weekofyear']).head(120))


## 14. San Juan 보정 해석

San Juan은 후보정 전 모델 기반 예측에서 큰 유행 구간을 충분히 키우지 못하거나, 피크 뒤쪽이 갑자기 끊기는 부분이 있었다.

2008년 week37/48은 반대로 낮춘 구간이다. 모델 예측이 높게 잡혀 있었지만, 제출 점수와 그래프 흐름을 확인했을 때 낮추는 쪽이 더 적절했다.

2010년 week25/36은 큰 outbreak 구간으로 보고 올렸다. 이후 week37/45는 피크 뒤쪽 tail로 보고 완만하게 이어지도록 조정했다. 2013년 week6/17도 같은 방식으로, 초반 높은 유행 뒤에 너무 급하게 떨어지는 부분을 tail 형태로 보정했다.

2012년 week22/52는 2013년 초 유행으로 이어지는 점진적 상승 구간으로 해석했다. 2012년 말이 너무 낮으면 2013년 초 유행이 갑자기 튀는 것처럼 보이기 때문에, 여러 주에 걸쳐 서서히 올라가도록 조정했다.


## 15. Iquitos 보정 해석

Iquitos는 San Juan과 다르게 처리했다. San Juan은 큰 outbreak가 중요한 도시지만, Iquitos는 전체 환자 수 규모가 작다.

처음에는 Iquitos에서도 연말과 연초의 고점 블록을 유행으로 보고 올리는 방향을 확인했다. 하지만 그래프상 2011년 말부터 2012년 초까지 이어지는 구간은 이미 높게 잡힌 블록처럼 보였고, 실제 제출 결과에서도 낮추는 방향이 더 좋았다.

따라서 이 구간은 한 주씩 따로 고치지 않고, 연속된 블록 전체를 낮추는 방식으로 정리했다. 이렇게 하면 특정 한 주만 튀는 모양을 만들지 않고, Iquitos의 작은 규모에 맞는 부드러운 흐름을 유지할 수 있다.


## 16. 최종 제출 파일 검증

마지막에는 예측값보다 제출 형식을 먼저 확인했다. 제출 파일은 대회에서 요구하는 4개 컬럼을 가져야 하고, 행 수와 순서가 제출 형식과 맞아야 한다. `total_cases`는 결측치가 없어야 하며, 음수가 아닌 정수여야 한다.


In [ ]:
def check_submission(sub):
    if sub is None:
        return pd.Series({'status': 'final_submission is missing'})

    required_cols = ['city', 'year', 'weekofyear', 'total_cases']

    checks = {
        'has_required_columns': list(sub.columns) == required_cols,
        'row_count': len(sub),
        'missing_total_cases': int(sub['total_cases'].isna().sum()),
        'min_total_cases': int(sub['total_cases'].min()),
        'is_integer_dtype': pd.api.types.is_integer_dtype(sub['total_cases']),
    }
    return pd.Series(checks)


submission_check = check_submission(final_submission)
display(submission_check)

if final_submission is not None:
    display(final_submission.describe(include='all'))


## 17. 최종 정리

처음에는 시간 순서를 유지한 validation과 tree ensemble 모델을 통해 약 26점대의 기준 성능을 만들었다. 이후 ExtraTrees를 중심으로 모델을 구성하고, 도시별 주차 평균을 섞어 후보정 전 모델 기반 예측을 만들었다.

최종 단계에서는 후보정 전 예측과 최종 예측을 비교하면서 시계열 흐름이 어색한 구간을 조정했다. San Juan은 2008년 과대예측 구간을 낮추고, 2010년 중반 outbreak와 tail, 2013년 초 tail, 2012년 후반 점진적 상승 구간을 보정했다. Iquitos는 2011년 말부터 2012년 초까지 이어지는 고점 블록을 도시 규모에 맞게 낮췄다.

최종 제출 파일 `dengai_assignment6_final_submission.csv`를 저장했고, 최종 Public score는 15.6563이었다.
